# Anggota 1 — Penyiapan Korpus Sistem Tanya Jawab

**Tugas Akhir Pemrosesan Bahasa Alami 2026 · QA Factoid tanpa LLM**

Notebook ini mengubah dokumen resmi FILKOM UB (PDF) menjadi **passage** — potongan teks yang akan dicari oleh sistem tanya jawab. Hasil akhirnya adalah `data/passages.jsonl`, yang menjadi masukan bagi Anggota 2 (pemrosesan pertanyaan), Anggota 3 (pencarian passage), dan Anggota 4 (penentuan jawaban).

| | |
|---|---|
| **Masukan** | `korpus_raw/*.pdf` (3 dokumen), `data/transkripsi_manual.json` (transkripsi tabel & rumus) |
| **Keluaran** | `data/passages.jsonl`, `data/dokumen.json`, `korpus_txt/*.txt` |
| **Menjalankan** | `uv sync`, lalu buka notebook ini dengan kernel `.venv` dan jalankan semua sel (*Run All*) |

### Alur pengolahan

```
PDF ──► 1. Ekstraksi baris (pdfplumber)
    ──► 2. Lewati sampul & daftar isi
    ──► 3. Buang nomor halaman, sisa rumus, kalimat penutup
    ──► 4. Normalisasi teks
    ──► 5. Tabel & rumus → diganti transkripsi manual
    ──► 6. Parsing hierarki (BAB → subbab → butir → sub-butir)
    ──► 7. Bentuk passage (+ pecah butir yang terlalu panjang)
    ──► 8. Simpan JSONL
    ──► 9. Validasi
```

## 0. Persiapan

Library yang dipakai:

- **`pdfplumber`** — membaca PDF dan memberi posisi setiap kata (koordinat `x0`, `top`). Posisi ini penting: indentasi dipakai untuk menentukan tingkat butir (tahap 6).
- **`re`** — seluruh aturan pembersihan dan pengenalan struktur ditulis sebagai *regular expression*.
- Sisanya pustaka standar Python (`json`, `dataclasses`, `pathlib`, `collections`).

Konstanta di bawah adalah parameter yang disepakati kelompok.

In [1]:
from __future__ import annotations

import json
import random
import re
import statistics
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import pdfplumber

DIR_KERJA = Path.cwd()
DIR_RAW = DIR_KERJA / "korpus_raw"
DIR_TXT = DIR_KERJA / "korpus_txt"
DIR_DATA = DIR_KERJA / "data"
DIR_TXT.mkdir(exist_ok=True)
DIR_DATA.mkdir(exist_ok=True)

BATAS_KATA = 150      # butir lebih panjang dari ini dipecah per sub-butir
TOLERANSI_X = 6.0     # selisih indentasi (pt) antar-penanda butir yang masih dianggap satu tingkat
TOLERANSI_LANJUTAN = 1.0   # baris lanjutan boleh sedikit lebih ke kiri dari penanda butirnya
TOLERANSI_Y = 5.0     # selisih posisi vertikal (pt) yang masih dianggap satu baris


def jumlah_kata(teks: str) -> int:
    return len(teks.split())

### Daftar dokumen korpus

Korpus inti yang disepakati kelompok: tiga dokumen dari halaman [Dokumen Resmi FILKOM](https://filkom.ub.ac.id/profil/dokumen-resmi/).

- `halaman_mulai` dan `pola_mulai` menandai awal isi sebenarnya (setelah sampul, surat keputusan, dan daftar isi).
- `pola_buang` berisi baris yang dibuang karena tidak memuat informasi, misalnya kalimat penutup surat.
- Dokumen `penilaian` adalah hasil **scan**: teks OCR bawaannya rusak, jadi isinya **diketik ulang manual** ke `data/transkripsi_manual.json` dan tidak diekstrak dari PDF.

In [2]:
DOKUMEN = {
    "pa2020": {
        "judul": "Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya",
        "tahun": 2020,
        "file": "Buku-Pedoman-Akademik-FILKOM-2020.pdf",
        "url": "https://filkom.ub.ac.id/wp-content/uploads/2021/09/Buku-Pedoman-Akademik-FILKOM-2020.pdf",
        "metode": "pdf",
        "halaman_mulai": 9,
        "pola_mulai": r"^VISI, MISI, DAN TUJUAN",
        "pola_buang": [],
    },
    "tatatertib": {
        "judul": "Tata Tertib Pelaksanaan Ujian (UTS & UAS) FILKOM UB",
        "tahun": 2016,
        "file": "Tata_Tertib_Pelaksanaan_UTS_dan_UAS_2.pdf",
        "url": "https://filkom.ub.ac.id/wp-content/uploads/2021/04/Tata_Tertib_Pelaksanaan_UTS_dan_UAS_2.pdf",
        "metode": "pdf",
        "halaman_mulai": 1,
        "pola_mulai": r"^PENGUMUMAN",
        "pola_buang": [r"^Demikian tata tertib"],
    },
    "penilaian": {
        "judul": "Pedoman Penilaian dan Perhitungan Indeks Prestasi PTIIK",
        "tahun": 2015,
        "file": "Pedoman_Penilaian_dan_Perhitungan_Indeks_Prestasi_PTIIK.pdf",
        "url": "https://filkom.ub.ac.id/wp-content/uploads/2021/04/Pedoman_Penilaian_dan_Perhitungan_Indeks_Prestasi_PTIIK.pdf",
        "metode": "manual",
    },
}

TRANSKRIPSI = json.loads((DIR_DATA / "transkripsi_manual.json").read_text(encoding="utf-8"))

DOKUMEN_PDF = [dok for dok, cfg in DOKUMEN.items() if cfg["metode"] == "pdf"]
print("Diekstrak dari PDF :", DOKUMEN_PDF)
print("Diketik ulang manual:", [d for d in DOKUMEN if d not in DOKUMEN_PDF])

Diekstrak dari PDF : ['pa2020', 'tatatertib']
Diketik ulang manual: ['penilaian']


## 1. Ekstraksi baris dari PDF

Kita tidak memakai `page.extract_text()` biasa karena dua masalah yang ditemukan saat memeriksa dokumen:

1. **Rumus matematika** (rumus Nilai Akhir dan Indeks Prestasi, hlm. 18 & 20) terekstrak menjadi puluhan baris sampah (`n`, `i=`, `B k`, `+`, …). Huruf-huruf rumus memakai font *Times New Roman Italic*, sedangkan isi dokumen memakai *Calibri*, sehingga karakter rumus bisa **disaring berdasarkan font**. Simbol sigma (``, font *Symbol*) juga dibuang. Rumusnya sendiri ditulis ulang secara manual pada tahap 5.
2. **Simbol ≥ dan ≤** ditulis dengan font *Symbol* yang posisinya sedikit lebih tinggi dari baris teks. Akibatnya `extract_text()` menaruh simbol itu di baris tersendiri, kadang di tempat yang salah. Contoh: `b. Mempunyai IPK ≥ 3,0` berubah menjadi `≥` di atas baris `b. Mempunyai IPK 3,0`.

Solusinya, baris disusun sendiri dari **kata-kata beserta koordinatnya**:

- kata diurutkan berdasarkan posisi vertikal (`top`), lalu horizontal (`x0`);
- kata yang selisih `top`-nya ≤ `TOLERANSI_Y` dianggap satu baris, sehingga simbol ≥ kembali menyatu dengan barisnya;
- `x0` kata pertama disimpan sebagai **indentasi baris**, yang dipakai pada tahap 6.

In [3]:
@dataclass
class Baris:
    teks: str
    x0: float   # indentasi (posisi horizontal kata pertama)
    hal: int    # nomor halaman PDF


def bukan_rumus(obj: dict) -> bool:
    # Buang karakter rumus: huruf variabel ber-font Times Italic dan simbol sigma.
    if obj.get("object_type") != "char":
        return True
    return "TimesNewRomanPS-Italic" not in obj.get("fontname", "") and obj["text"] != ""


def ekstrak_baris_halaman(halaman: pdfplumber.page.Page, nomor: int) -> list[Baris]:
    kata = halaman.filter(bukan_rumus).extract_words()
    kata.sort(key=lambda k: (k["top"], k["x0"]))

    kelompok: list[list[dict]] = []
    for k in kata:
        if kelompok and abs(k["top"] - kelompok[-1][0]["top"]) <= TOLERANSI_Y:
            kelompok[-1].append(k)
        else:
            kelompok.append([k])

    hasil = []
    for anggota in kelompok:
        anggota.sort(key=lambda k: k["x0"])
        teks = " ".join(k["text"] for k in anggota)
        hasil.append(Baris(teks=teks, x0=round(anggota[0]["x0"], 1), hal=nomor))
    return hasil


def ekstrak_pdf(path: Path) -> list[Baris]:
    with pdfplumber.open(path) as pdf:
        return [b for i, hlm in enumerate(pdf.pages, start=1) for b in ekstrak_baris_halaman(hlm, i)]


def simpan_txt(dok: str, baris: list[Baris]) -> Path:
    # Simpan teks mentah per halaman agar mudah dibandingkan dengan PDF.
    keluaran, hal_terakhir = [], None
    for b in baris:
        if b.hal != hal_terakhir:
            keluaran.append(f"\n<<HAL {b.hal}>>")
            hal_terakhir = b.hal
        keluaran.append(b.teks)
    path = DIR_TXT / f"{dok}.txt"
    path.write_text("\n".join(keluaran).strip() + "\n", encoding="utf-8")
    return path


baris_mentah = {}
for dok in DOKUMEN_PDF:
    baris_mentah[dok] = ekstrak_pdf(DIR_RAW / DOKUMEN[dok]["file"])
    path = simpan_txt(dok, baris_mentah[dok])
    jml_hal = baris_mentah[dok][-1].hal
    print(f"{dok:<11} {jml_hal:>3} halaman, {len(baris_mentah[dok]):>5} baris  -> {path.relative_to(DIR_KERJA)}")

pa2020       47 halaman,  1759 baris  -> korpus_txt\pa2020.txt


tatatertib    3 halaman,    76 baris  -> korpus_txt\tatatertib.txt


Contoh hasil: baris `b. Mempunyai IPK ≥ 3,0` (hlm. 26). Simbol `≥` masih berupa karakter *private use* `` dan baru diganti pada tahap 4.

In [4]:
for b in baris_mentah["pa2020"]:
    if b.hal == 26 and "Mempunyai IPK" in b.teks:
        print(b)

Baris(teks='b. Mempunyai IPK \uf0b3 3,0.', x0=121.1, hal=26)


## 2. Melewati sampul, surat keputusan, dan daftar isi

Halaman 1–8 Pedoman FILKOM berisi sampul, Peraturan Dekan, daftar isi, kata pengantar, dan tim penyusun. Bagian ini dibuang karena:

- **daftar isi** berisi baris seperti `BAB III ADMINISTRASI PENDIDIKAN .......... 24`, yang bisa "mencuri" hasil pencarian karena memuat kata kunci tanpa isi;
- **daftar nama tim penyusun** tidak relevan untuk pertanyaan akademik.

Isi dimulai dari halaman *Visi, Misi, dan Tujuan* (hlm. 9), karena halaman ini dan *Sejarah Singkat* (hlm. 10) memuat fakta yang layak ditanyakan, misalnya tahun berdirinya FILKOM.

In [5]:
def potong_awal(baris: list[Baris], cfg: dict) -> list[Baris]:
    kandidat = [b for b in baris if b.hal >= cfg["halaman_mulai"]]
    pola = re.compile(cfg["pola_mulai"])
    for i, b in enumerate(kandidat):
        if pola.search(b.teks):
            return kandidat[i:]
    raise ValueError(f"Pola awal {cfg['pola_mulai']!r} tidak ditemukan")


baris_isi = {dok: potong_awal(baris_mentah[dok], DOKUMEN[dok]) for dok in DOKUMEN_PDF}
for dok in DOKUMEN_PDF:
    dibuang = len(baris_mentah[dok]) - len(baris_isi[dok])
    print(f"{dok:<11} dibuang {dibuang:>3} baris awal, mulai: {baris_isi[dok][0].teks!r} (hlm. {baris_isi[dok][0].hal})")

pa2020      dibuang 165 baris awal, mulai: 'VISI, MISI, DAN TUJUAN' (hlm. 9)
tatatertib  dibuang   0 baris awal, mulai: 'PENGUMUMAN' (hlm. 1)


## 3. Membuang baris yang bukan isi

| Jenis | Aturan | Alasan |
|---|---|---|
| **Nomor halaman** | baris yang isinya hanya angka **sama dengan nomor halaman PDF** | Nomor halaman bisa muncul di tengah teks (mis. hlm. 15, di dalam tabel). Membandingkan dengan nomor halaman, bukan membuang semua baris berisi angka, mencegah terbuangnya angka penting. |
| **Sisa rumus** | baris yang hanya berisi `= + .`, spasi, dan digit | Setelah huruf rumus disaring di tahap 1, operatornya masih tersisa (`= = 1`, `. +`). |
| **Pola buang** | regex per dokumen (`pola_buang`) | Kalimat penutup seperti *"Demikian tata tertib ini dibuat …"* tidak berisi informasi. |

In [6]:
POLA_SISA_RUMUS = re.compile(r"[=+.\s\d]+")
POLA_PENANDA_SAJA = re.compile(r"\d{1,2}\.")


def jenis_sampah(b: Baris, pola_buang: list[re.Pattern]) -> str | None:
    teks = b.teks.strip()
    if teks == str(b.hal):
        return "nomor halaman"
    if POLA_SISA_RUMUS.fullmatch(teks) and not POLA_PENANDA_SAJA.fullmatch(teks):
        return "sisa rumus"
    if any(p.search(teks) for p in pola_buang):
        return "pola buang"
    return None


def buang_sampah(baris: list[Baris], cfg: dict) -> tuple[list[Baris], Counter]:
    pola_buang = [re.compile(p) for p in cfg["pola_buang"]]
    bersih, statistik = [], Counter()
    for b in baris:
        jenis = jenis_sampah(b, pola_buang)
        if jenis:
            statistik[jenis] += 1
        else:
            bersih.append(b)
    return bersih, statistik


baris_bersih = {}
for dok in DOKUMEN_PDF:
    baris_bersih[dok], statistik = buang_sampah(baris_isi[dok], DOKUMEN[dok])
    print(f"{dok:<11} {dict(statistik)}")

pa2020      {'nomor halaman': 39, 'sisa rumus': 7}
tatatertib  {'pola buang': 1}


## 4. Normalisasi teks

Normalisasi dibuat **sekecil mungkin**: tujuannya memperbaiki kerusakan akibat ekstraksi, **bukan** mengubah isi dokumen. Teks tetap disimpan dalam huruf aslinya (tanpa *case folding*, *stemming*, atau penghapusan stopword) karena:

- Anggota 4 mengekstrak jawaban seperti `IPK ≥ 2,00` atau `144 sks` langsung dari teks passage;
- *preprocessing* untuk pencarian (tokenisasi, stemming) dilakukan Anggota 3 dengan fungsi yang **sama** seperti untuk pertanyaan.

| Aturan | Contoh sebelum → sesudah |
|---|---|
| Karakter *private use* font Symbol | `` → `≥`, `` → `≤` |
| Huruf `l` kecil yang seharusnya `I` | `lPK` → `IPK`, `lndeks` → `Indeks` |
| Huruf `l` yang seharusnya angka `1` | `boleh l (satu) kali` → `boleh 1 (satu) kali` |
| Spasi di sekitar simbol pembanding | `IPK≥3,0` → `IPK ≥ 3,0` |
| Spasi setelah tanda hubung kata ulang | `sekurang- kurangnya` → `sekurang-kurangnya` |
| Spasi sebelum tanda baca | `laporan kegiatan .` → `laporan kegiatan.` |
| Spasi ganda | dirapikan menjadi satu spasi |

Typo dari penulis dokumen (misalnya `Pelaksanan`) **tidak** diperbaiki, karena itu bagian dari dokumen asli.

In [7]:
PETA_SIMBOL = str.maketrans({"": "≥", "": "≤"})

PERBAIKAN_EKSTRAKSI = [
    (re.compile(r"\blPK\b"), "IPK"),
    (re.compile(r"\blP\b"), "IP"),
    (re.compile(r"\blndeks\b"), "Indeks"),
    (re.compile(r"\bIImu\b"), "Ilmu"),
    (re.compile(r"\bl \((satu)\)"), r"1 (\1)"),
]
POLA_SIMBOL_PEMBANDING = re.compile(r"\s*([≥≤])\s*")
POLA_SPASI_KATA_ULANG = re.compile(r"(?<=[a-z])- (?=[a-z])")
POLA_SPASI_SEBELUM_TANDA_BACA = re.compile(r"\s+([.,;:])")
POLA_SPASI_GANDA = re.compile(r"\s{2,}")


def normalisasi(teks: str) -> str:
    teks = teks.translate(PETA_SIMBOL)
    for pola, ganti in PERBAIKAN_EKSTRAKSI:
        teks = pola.sub(ganti, teks)
    teks = POLA_SIMBOL_PEMBANDING.sub(r" \1 ", teks)
    teks = POLA_SPASI_KATA_ULANG.sub("-", teks)
    teks = POLA_SPASI_SEBELUM_TANDA_BACA.sub(r"\1", teks)
    teks = POLA_SPASI_GANDA.sub(" ", teks)
    return teks.strip()


baris_normal, contoh_perubahan = {}, []
for dok in DOKUMEN_PDF:
    baris_normal[dok] = []
    for b in baris_bersih[dok]:
        baru = normalisasi(b.teks)
        if baru != b.teks:
            contoh_perubahan.append((dok, b.hal, b.teks, baru))
        if baru:
            baris_normal[dok].append(Baris(baru, b.x0, b.hal))

print(f"{len(contoh_perubahan)} baris berubah. Contoh:")
for dok, hal, lama, baru in contoh_perubahan[:8]:
    print(f"  [{dok} hlm.{hal}]\n    - {lama[:90]}\n    + {baru[:90]}")

54 baris berubah. Contoh:
  [pa2020 hlm.14]
    - 1. Fakultas Ilmu Komputer menerapkan kurikulum yang mengacu pada :
    + 1. Fakultas Ilmu Komputer menerapkan kurikulum yang mengacu pada:
  [pa2020 hlm.14]
    - sebagai berikut :
    + sebagai berikut:
  [pa2020 hlm.15]
    - kewajiban pembuatan laporan kegiatan .
    + kewajiban pembuatan laporan kegiatan.
  [pa2020 hlm.15]
    - IP  3,00 24 sks
    + IP ≥ 3,00 24 sks
  [pa2020 hlm.15]
    - 2,50  IP < 3,00 21 sks
    + 2,50 ≤ IP < 3,00 21 sks
  [pa2020 hlm.15]
    - 2,00  IP < 2,50 18 sks
    + 2,00 ≤ IP < 2,50 18 sks
  [pa2020 hlm.15]
    - 1,50  IP < 2,00 15 sks
    + 1,50 ≤ IP < 2,00 15 sks
  [pa2020 hlm.17]
    - kecuali bila mahasiswa :
    + kecuali bila mahasiswa:


## 5. Tabel dan rumus: diganti transkripsi manual

### 5a. Tabel

Saat diekstrak, struktur tabel hilang. Contoh *Tabel 1* (hlm. 15) menjadi:

```
IP ≥ 3,00 24 sks
2,50 ≤ IP < 3,00 21 sks
```

Baris seperti ini sulit dicari dan sulit diekstrak jawabannya. Karena dokumen hanya berisi **empat tabel**, setiap tabel diketik ulang di `data/transkripsi_manual.json` (sudah dicek terhadap PDF), lalu:

1. baris tabel dihapus dari aliran teks, mulai dari pola `mulai` sampai sebelum pola `sampai`;
2. **setiap baris tabel menjadi satu passage berbentuk kalimat** menggunakan `templat`, misalnya:
   > *Jika Indeks Prestasi (IP) semester sebelumnya 2,50 ≤ IP < 3,00, beban studi maksimum yang boleh diambil pada semester berikutnya adalah 21 sks.*

Pencarian berbasis kata bekerja jauh lebih baik pada kalimat lengkap daripada pada baris tabel mentah.

### 5b. Rumus

Rumus **Nilai Akhir** (hlm. 18) dan **Indeks Prestasi** (hlm. 20) adalah objek rumus yang terbuang pada tahap 1. Keduanya ditulis ulang manual sebagai teks satu baris, lengkap dengan **penjelasan dalam kalimat biasa**, lalu disisipkan tepat setelah baris pengantarnya (pola `setelah`), misalnya setelah *"3. Perhitungan Indeks Prestasi dilakukan sebagai berikut:"*.

Baris sisipan diberi indentasi sedikit lebih dalam dari baris pengantarnya, supaya pada tahap 6 terbaca sebagai **lanjutan butir yang sama**. Penjelasan kalimatnya penting untuk pencarian: pertanyaan *"Bagaimana cara menghitung IP?"* memakai kata seperti *menghitung, kredit, nilai*, bukan simbol `Σ`.

In [8]:
def potong_tabel(baris: list[Baris], daftar_tabel: list[dict]) -> list[Baris]:
    for tabel in daftar_tabel:
        mulai = next(i for i, b in enumerate(baris) if re.search(tabel["mulai"], b.teks))
        sampai = next(i for i in range(mulai + 1, len(baris)) if re.search(tabel["sampai"], baris[i].teks))
        dibuang = baris[mulai:sampai]
        print(f"  {tabel['id']:<7} hlm.{dibuang[0].hal}: {len(dibuang):>2} baris dibuang "
              f"({dibuang[0].teks[:40]!r} … {dibuang[-1].teks[:25]!r})")
        baris = baris[:mulai] + baris[sampai:]
    return baris


def sisipkan_rumus(baris: list[Baris], daftar_rumus: list[dict]) -> list[Baris]:
    for rumus in daftar_rumus:
        i = next(i for i, b in enumerate(baris) if re.search(rumus["setelah"], b.teks))
        pengantar = baris[i]
        sisipan = Baris(rumus["teks"], x0=pengantar.x0 + 18, hal=rumus["halaman"])   # lanjutan butir yang sama
        print(f"  {rumus['id']:<16} disisipkan setelah {pengantar.teks[:55]!r} (hlm.{pengantar.hal})")
        baris = baris[: i + 1] + [sisipan] + baris[i + 1:]
    return baris


baris_final = {}
for dok in DOKUMEN_PDF:
    manual = TRANSKRIPSI.get(dok, {})
    if manual:
        print(dok)
    baris_final[dok] = potong_tabel(baris_normal[dok], manual.get("tabel", []))
    baris_final[dok] = sisipkan_rumus(baris_final[dok], manual.get("rumus", []))

pa2020
  tabel1  hlm.15:  8 baris dibuang ('Tabel 1. Beban Studi maksimal berdasarka' … 'IP < 1,50 < 12 sks')
  tabel2  hlm.18: 12 baris dibuang ('Tabel 2 Hasil penilaian akhir mata kulia' … 'E 0 Sangat Kurang')
  tabel3  hlm.19: 10 baris dibuang ('Tabel 3. Konversi angka ke Huruf' … '0 ≤ N ≤ 44 E')
  bobot   hlm.19:  9 baris dibuang ('Nilai Huruf Bobot' … 'E 0,0')
  nilai_akhir      disisipkan setelah '4. Sebagai pedoman, Nilai Akhir masing-masing mata kuli' (hlm.18)
  indeks_prestasi  disisipkan setelah '3. Perhitungan Indeks Prestasi dilakukan sebagai beriku' (hlm.20)


## 6. Parsing hierarki dokumen

Struktur Pedoman FILKOM:

```
BAB II  SISTEM PENDIDIKAN                    ← BAB (judul di baris berikutnya)
   paragraf pembuka bab
   B. NILAI KREDIT                           ← subbab (huruf kapital + judul KAPITAL)
      1. Nilai Kredit Semester untuk ...     ← butir (angka)
         a. Untuk Mahasiswa                  ← sub-butir (huruf kecil)
            i. Lima puluh menit ...          ← sub-sub-butir (romawi kecil)
```

Setiap baris diklasifikasikan dengan regex:

| Jenis baris | Pola | Contoh |
|---|---|---|
| BAB | `^BAB\s+([IVX]+)$` | `BAB II` |
| Subbab | `^([A-Z])\.\s+(.+)$` dan seluruh judul huruf kapital | `B. NILAI KREDIT` |
| Butir | `^(\d{1,2}\|[a-z]\|[ivx]{1,4})\.\s+(.*)$` | `7. Seorang mahasiswa …`, `a. IPK ≥ 2,00.` |
| Judul tanpa nomor | seluruhnya huruf kapital | `SEJARAH SINGKAT`, `SANKSI AKADEMIK` |
| Lanjutan | selain di atas | sambungan kalimat baris sebelumnya |

### Mengapa memakai indentasi, bukan hanya nomor?

Awalnya tingkat butir mau ditentukan dari urutan nomornya saja (butir `8` diikuti `9`, dst.). Ternyata ada **daftar bernomor di dalam butir bernomor**. Contoh subbab J (hlm. 29):

```
1. Perpindahan Mahasiswa di lingkungan Universitas Brawijaya      x0 = 85.1
   a. Perpindahan program studi dalam satu fakultas               x0 = 103.1
   1. Mahasiswa yang dapat mengajukan pindah ...                  x0 = 102.5   ← nomor mulai lagi
   2. Bukan gagal studi ...                                       x0 = 102.5
   …
   11. Permohonan pindah harus diterima Rektor ...                x0 = 102.5
2. Perpindahan Mahasiswa dari PTN lain ke Universitas Brawijaya   x0 = 85.1
```

Dengan urutan nomor saja, `2. Bukan gagal studi` akan salah dianggap butir ke-2 subbab J. Karena itu tingkat ditentukan dari **indentasi (`x0`)** dengan sebuah **stack** (mirip *pushdown automaton*):

- butir baru yang lebih menjorok dari puncak stack → **anak**;
- indentasinya sama (selisih ≤ `TOLERANSI_X`) → **saudara**;
- lebih ke kiri → keluar (*pop*) sampai menemukan tingkat yang sesuai.

Baris lanjutan juga memakai indentasi. Baris lanjutan milik butir terdalam jika letaknya tidak lebih ke kiri dari penanda butir itu (toleransi `TOLERANSI_LANJUTAN` = 1 pt; baris sambungan biasanya menjorok **lebih dalam** dari penandanya). Jika lebih ke kiri, baris itu milik butir induknya. Contoh di subbab J:

```
1. Perpindahan Mahasiswa di lingkungan Universitas Brawijaya         x0 =  85.1
   Merupakan perpindahan ... terdiri dari :                          x0 =  99.3   ← milik butir 1
   a. ...                                                            x0 = 103.1
   b. Perpindahan program studi antar fakultas                       x0 = 103.1
   Adapun persyaratan dan prosedur ... antara lain:                  x0 =  99.3   ← milik butir 1, bukan b.
   1. Mahasiswa yang dapat mengajukan pindah ...                     x0 = 102.5
```

Isi sebuah butir disimpan sebagai **daftar berurutan** berisi teks dan butir anak (`isi`), sehingga kalimat seperti *"Adapun persyaratan … antara lain:"* tetap berada **di antara** sub-butir `b.` dan daftar `1.`–`11.`, sesuai urutan aslinya.

Judul tanpa nomor (`SEJARAH SINGKAT`, `SANKSI AKADEMIK`) dikenali dari huruf kapital semua **dan diakhiri huruf**. Syarat terakhir mencegah baris lanjutan pendek seperti `IPK:` atau `PKL).` salah dianggap judul.

In [9]:
POLA_BAB = re.compile(r"^BAB\s+([IVX]+)$")
POLA_SUBBAB = re.compile(r"^([A-Z])\.\s+(.+)$")
POLA_BUTIR = re.compile(r"^(\d{1,2}|[a-z]|[ivx]{1,4})\.\s+(.*)$")


def huruf_kapital_semua(teks: str) -> bool:
    huruf = [c for c in teks if c.isalpha()]
    return len(huruf) >= 3 and all(c.isupper() for c in huruf)


def klasifikasi(teks: str) -> tuple[str, re.Match | None]:
    if m := POLA_BAB.match(teks):
        return "bab", m
    if (m := POLA_SUBBAB.match(teks)) and huruf_kapital_semua(m[2]):
        return "subbab", m
    if m := POLA_BUTIR.match(teks):
        return "butir", m
    if huruf_kapital_semua(teks) and teks[-1].isalpha():
        return "judul", None
    return "lanjutan", None


@dataclass
class Butir:
    penanda: str
    x0: float
    hal: int
    isi: list["str | Butir"] = field(default_factory=list)   # teks dan butir anak, berurutan

    @property
    def anak(self) -> list["Butir"]:
        return [x for x in self.isi if isinstance(x, Butir)]


@dataclass
class Bagian:
    kode: str           # huruf subbab, atau S1, S2, ... untuk judul tanpa huruf, atau 0
    bab: str
    judul_bab: str
    subbab: str
    judul_subbab: str
    hal: int
    paragraf: list[tuple[int, str]] = field(default_factory=list)   # teks tanpa nomor sebelum butir pertama
    butir: list[Butir] = field(default_factory=list)


def gabung_baris(daftar: list[str]) -> str:
    # Gabungkan baris; baris berakhiran '-' + huruf kecil adalah kata ulang yang terpotong.
    teks = ""
    for baris in daftar:
        if teks.endswith("-") and baris[:1].islower():
            teks += baris            # syarat- + syarat -> syarat-syarat
        else:
            teks = f"{teks} {baris}".strip()
    return teks


def segmen(butir: Butir) -> list[str | Butir]:
    # Isi butir dengan baris-baris teks yang berurutan sudah digabung menjadi satu teks.
    hasil: list[str | Butir] = []
    kumpulan: list[str] = []
    for x in butir.isi:
        if isinstance(x, str):
            kumpulan.append(x)
            continue
        if kumpulan:
            hasil.append(gabung_baris(kumpulan))
            kumpulan = []
        hasil.append(x)
    if kumpulan:
        hasil.append(gabung_baris(kumpulan))
    return hasil

Fungsi parser memproses baris satu per satu dan menyimpan **status**: bagian yang sedang aktif, stack butir terbuka, dan apakah baris sebelumnya adalah judul (judul bisa terdiri dari beberapa baris, misalnya judul BAB IV).

In [10]:
class ParserDokumen:
    def __init__(self) -> None:
        self.bagian: list[Bagian] = []
        self.stack: list[Butir] = []
        self.bab = ""
        self.judul_bab = ""
        self.mode_judul: str | None = None   # "bab" / "subbab" saat judul masih bisa berlanjut
        self.nomor_judul = 0
        self.audit = Counter()

    # ---- pembuatan bagian ----
    def _bagian_baru(self, kode: str, subbab: str, judul: str, hal: int) -> None:
        self.bagian.append(Bagian(kode, self.bab, self.judul_bab, subbab, judul, hal))
        self.stack = []

    @property
    def aktif(self) -> Bagian:
        return self.bagian[-1]

    # ---- penanganan tiap jenis baris ----
    def _bab(self, m: re.Match, b: Baris) -> None:
        self.bab, self.judul_bab = m[1], ""
        self._bagian_baru("0", "", "", b.hal)
        self.mode_judul = "bab"

    def _subbab(self, m: re.Match, b: Baris) -> None:
        self._bagian_baru(m[1], m[1], m[2], b.hal)
        self.mode_judul = "subbab"

    def _judul(self, b: Baris) -> None:
        if self.mode_judul == "bab":
            self.judul_bab = f"{self.judul_bab} {b.teks}".strip()
            self.aktif.judul_bab = self.judul_bab
            return
        satu_kata = len(b.teks.split()) == 1
        if self.mode_judul == "subbab" and not satu_kata:
            self.aktif.judul_subbab = f"{self.aktif.judul_subbab} {b.teks}"
            return
        self.nomor_judul += 1
        self._bagian_baru(f"S{self.nomor_judul}", "", b.teks, b.hal)
        self.mode_judul = "subbab"

    def _butir(self, m: re.Match, b: Baris) -> None:
        baru = Butir(m[1], b.x0, b.hal, [m[2]])
        while self.stack and self.stack[-1].x0 > b.x0 + TOLERANSI_X:   # lebih dalam -> tutup
            self.stack.pop()
        if self.stack and abs(self.stack[-1].x0 - b.x0) <= TOLERANSI_X:  # setingkat -> saudara
            self.stack.pop()
        if self.stack:
            self.stack[-1].isi.append(baru)
        else:
            self.aktif.butir.append(baru)
        self.stack.append(baru)

    def _lanjutan(self, b: Baris) -> None:
        while self.stack and self.stack[-1].x0 > b.x0 + TOLERANSI_LANJUTAN:   # lebih ke kiri -> milik induk
            self.stack.pop()
        if self.stack:
            target = self.stack[-1]
        elif self.aktif.butir:                     # teks setelah daftar berakhir -> butir terakhir
            target = self.aktif.butir[-1]
            self.audit["lanjutan setelah daftar"] += 1
        else:
            self.aktif.paragraf.append((b.hal, b.teks))
            self.audit["paragraf"] += 1
            return
        target.isi.append(b.teks)

    # ---- loop utama ----
    def parse(self, baris: list[Baris]) -> list[Bagian]:
        self._bagian_baru("0", "", "", baris[0].hal)
        for b in baris:
            jenis, m = klasifikasi(b.teks)
            self.audit[jenis] += 1
            if jenis != "judul":
                self.mode_judul = None if jenis in ("butir", "lanjutan") else self.mode_judul
            if jenis == "bab":
                self._bab(m, b)
            elif jenis == "subbab":
                self._subbab(m, b)
            elif jenis == "judul":
                self._judul(b)
            elif jenis == "butir":
                self._butir(m, b)
            else:
                self._lanjutan(b)
        return [bg for bg in self.bagian if bg.paragraf or bg.butir]


hasil_parse, audit_parse = {}, {}
for dok in DOKUMEN_PDF:
    parser = ParserDokumen()
    hasil_parse[dok] = parser.parse(baris_final[dok])
    audit_parse[dok] = parser.audit
    print(f"{dok:<11} {len(hasil_parse[dok]):>3} bagian berisi | klasifikasi baris: {dict(parser.audit)}")

pa2020       56 bagian berisi | klasifikasi baris: {'judul': 15, 'lanjutan': 758, 'paragraf': 72, 'butir': 685, 'bab': 7, 'subbab': 46}
tatatertib    2 bagian berisi | klasifikasi baris: {'judul': 3, 'butir': 28, 'lanjutan': 44, 'paragraf': 4}


### Melihat pohon hasil parsing

Untuk memastikan parser bekerja, cetak pohon subbab J (perpindahan mahasiswa), kasus tersulit dengan nomor yang mulai lagi dari 1.

In [11]:
def ringkas(teks: str, n: int) -> str:
    return teks if len(teks) <= n else teks[:n] + "…"


def cetak_pohon(daftar: list[Butir], kedalaman: int = 0, maks_kar: int = 70) -> None:
    indent = "    " * kedalaman
    for butir in daftar:
        for i, x in enumerate(segmen(butir)):
            if isinstance(x, Butir):
                cetak_pohon([x], kedalaman + 1, maks_kar)
            elif i == 0:
                print(f"{indent}{butir.penanda}. {ringkas(x, maks_kar)}")
            else:
                print(f"{indent}    [teks] {ringkas(x, maks_kar)}")


def cari_bagian(dok: str, bab: str, kode: str) -> Bagian:
    return next(bg for bg in hasil_parse[dok] if bg.bab == bab and bg.kode == kode)


bagian_j = cari_bagian("pa2020", "III", "J")
print(f"BAB {bagian_j.bab} {bagian_j.judul_bab} / {bagian_j.subbab}. {bagian_j.judul_subbab}\n")
cetak_pohon(bagian_j.butir)

BAB III ADMINISTRASI PENDIDIKAN / J. PERPINDAHAN MAHASISWA

1. Perpindahan Mahasiswa di lingkungan Universitas Brawijaya Merupakan pe…
    a. Perpindahan program studi dalam satu fakultas
    b. Perpindahan program studi antar fakultas
    [teks] Adapun persyaratan dan prosedur yang harus dipenuhi dalam pengajuan pe…
    1. Mahasiswa yang dapat mengajukan pindah program studi adalah telah meng…
        a. Untuk 2 semester, 24 sks dengan IPK sekurang-kurangnya 2,75.
        b. Untuk 4 semester, 48 sks dengan IPK sekurang-kurangnya 2,75.
    2. Bukan gagal studi karena tidak memenuhi ketentuan akademik pada progra…
    3. Tidak pernah melanggar peraturan pada program studi asal.
    4. Perpindahan harus dalam jenjang pendidikan yang sama/setara dan status…
    5. Permohonan pindah program studi baik di dalam/antar fakultas diajukan …
    6. Jika permohonan telah disetujui di fakultas, maka Dekan/Direktur/ Ketu…
    7. Persetujuan dan kesediaan Dekan/Direktur/Ketua pada program studi yang

In [12]:
print("Daftar bagian Pedoman FILKOM:")
for bg in hasil_parse["pa2020"]:
    print(f"  BAB {bg.bab or '-':<4} {bg.kode:<3} {bg.judul_subbab[:55]:<55} "
          f"{len(bg.paragraf):>2} paragraf, {len(bg.butir):>2} butir")

Daftar bagian Pedoman FILKOM:
  BAB -    S2  VISI                                                     3 paragraf,  0 butir
  BAB -    S3  MISI                                                     0 paragraf,  4 butir
  BAB -    S4  TUJUAN                                                   0 paragraf,  4 butir
  BAB -    S5  SEJARAH SINGKAT BERDIRINYA FAKULTAS ILMU KOMPUTER       27 paragraf,  0 butir
  BAB I    0                                                            4 paragraf, 29 butir
  BAB II   0                                                            6 paragraf,  0 butir
  BAB II   A   TUJUAN PENERAPAN KURIKULUM STANDAR NASIONAL PENDIDIKAN   0 paragraf,  3 butir
  BAB II   B   NILAI KREDIT                                             0 paragraf,  2 butir
  BAB II   C   BEBAN STUDI MAHASISWA                                    1 paragraf,  4 butir
  BAB II   D   PELAKSANAAN PEMBELAJARAN                                 0 paragraf,  5 butir
  BAB II   E   PEMBELAJARAN DARING      

## 7. Membentuk passage

**Unit passage = satu butir angka beserta seluruh sub-butirnya.** Alasannya: sub-butir seperti `a. IPK ≥ 2,00.` tidak bermakna tanpa kalimat induknya (*"Seorang mahasiswa dinyatakan telah selesai … dengan syarat-syarat sebagai berikut:"*).

Aturan tambahan:

1. **Butir yang panjangnya > `BATAS_KATA` kata dipecah per anak.** Kalimat pembuka butir induk dipindahkan ke field `konteks`, supaya setiap pecahan tetap punya konteks. Jika di antara anak-anak ada kalimat pengantar (mis. *"Adapun persyaratan … antara lain:"*), kalimat itu ikut menjadi konteks bagi anak-anak sesudahnya. Pemecahan berjalan rekursif jika anaknya masih terlalu panjang.
2. **Kalimat pengantar daftar.** Paragraf pembuka bagian yang diakhiri titik dua (mis. *"Untuk melaksanakan kegiatan pendidikan diperlukan administrasi akademik yang meliputi:"*) tidak dijadikan passage sendiri. Kalimat terakhirnya menjadi `konteks` bagi setiap butir di bawahnya. Tanpa aturan ini, passage seperti `2. Penentuan rencana studi semester mahasiswa.` kehilangan maknanya.
3. **Paragraf tanpa nomor** lainnya (pembuka BAB, Visi, Sejarah) menjadi passage `jenis: "paragraf"`. Jika terlalu panjang, paragraf dipecah per kalimat (*sentence splitting*), lalu kalimat dikumpulkan sampai mendekati `BATAS_KATA`.
4. **Penggabungan baris** memakai `gabung_baris` (tahap 6): baris yang berakhir dengan `-` lalu diikuti huruf kecil digabung **tanpa spasi dan tanpa membuang tanda hubung**. Dalam bahasa Indonesia, kasus ini hampir selalu **kata ulang** (`syarat-` + `syarat` → `syarat-syarat`), berbeda dengan bahasa Inggris (`infor-` + `mation` → `information`).

Pemecahan kalimat tidak bisa sekadar memotong di setiap titik, karena dokumen penuh singkatan bertitik: *"Prof. Dr. Ir. Yogi Sugito"*, *"No. 20 tahun 2003"*. Potongan yang diakhiri singkatan digabung kembali dengan potongan berikutnya.

In [13]:
SINGKATAN = {"No", "Prof", "Dr", "Ir", "Eng", "dll", "dsb", "dkk", "Jl", "JL", "Telp", "S", "M", "T", "Kom"}
POLA_BATAS_KALIMAT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9“\"(])")


def pecah_kalimat(teks: str) -> list[str]:
    kalimat: list[str] = []
    for bagian_teks in POLA_BATAS_KALIMAT.split(teks):
        kata_terakhir = kalimat[-1].rstrip(".").split()[-1] if kalimat else ""
        if kalimat and kata_terakhir.split(".")[-1] in SINGKATAN:
            kalimat[-1] += " " + bagian_teks   # titik milik singkatan, bukan akhir kalimat
        else:
            kalimat.append(bagian_teks)
    return kalimat


def kelompokkan_kalimat(kalimat: list[str], batas: int = BATAS_KATA) -> list[str]:
    kelompok, sekarang = [], []
    for k in kalimat:
        if sekarang and jumlah_kata(" ".join(sekarang + [k])) > batas:
            kelompok.append(" ".join(sekarang))
            sekarang = []
        sekarang.append(k)
    if sekarang:
        kelompok.append(" ".join(sekarang))
    return kelompok


def teks_sendiri(butir: Butir) -> str:
    # Penanda + kalimat pembuka butir (tanpa anak-anaknya).
    pertama = segmen(butir)[0]
    return f"{butir.penanda}. {pertama}" if isinstance(pertama, str) else f"{butir.penanda}."


def teks_lengkap(butir: Butir) -> str:
    potongan = [f"{butir.penanda}."]
    for x in segmen(butir):
        potongan.append(teks_lengkap(x) if isinstance(x, Butir) else x)
    return " ".join(potongan)


print(pecah_kalimat("Pada bulan Oktober 2011, Prof. Dr. Ir. Yogi Sugito selaku rektor mengeluarkan SK. Selanjutnya PTIIK berdiri."))
print(gabung_baris(["memenuhi syarat-", "syarat sebagai berikut:"]))

['Pada bulan Oktober 2011, Prof. Dr. Ir. Yogi Sugito selaku rektor mengeluarkan SK.', 'Selanjutnya PTIIK berdiri.']
memenuhi syarat-syarat sebagai berikut:


In [14]:
def buat_passage(dok: str, bg: Bagian, butir_id: str, teks: str, konteks: list[str], hal: int, jenis: str) -> dict:
    kode_bab = bg.bab or "X"
    return {
        "id": f"{dok}-{kode_bab}-{bg.kode}-{butir_id}",
        "dok_id": dok,
        "halaman": hal,
        "bab": bg.bab,
        "judul_bab": bg.judul_bab,
        "subbab": bg.subbab,
        "judul_subbab": bg.judul_subbab,
        "butir": butir_id if jenis == "butir" else "",
        "konteks": " › ".join(konteks),
        "teks": teks,
        "jenis": jenis,
    }


def passage_dari_butir(dok: str, bg: Bagian, butir: Butir, jalur: list[str], konteks: list[str]) -> list[dict]:
    jalur = jalur + [butir.penanda]
    lengkap = teks_lengkap(butir)
    if jumlah_kata(lengkap) <= BATAS_KATA or not butir.anak:
        return [buat_passage(dok, bg, ".".join(jalur), lengkap, konteks, butir.hal, "butir")]

    # Terlalu panjang: kalimat pembuka induk menjadi konteks, setiap anak menjadi passage sendiri.
    konteks_induk = konteks + [teks_sendiri(butir)]
    isi = segmen(butir)
    if isinstance(isi[0], str):
        isi = isi[1:]                       # kalimat pembuka sudah masuk konteks
    hasil, pengantar, nomor_teks = [], None, 0
    for i, x in enumerate(isi):
        if isinstance(x, Butir):
            k = konteks_induk + ([pengantar] if pengantar else [])
            hasil += passage_dari_butir(dok, bg, x, jalur, k)
        elif any(isinstance(y, Butir) for y in isi[i + 1:]):
            pengantar = x                   # kalimat pengantar untuk anak-anak sesudahnya
        else:
            nomor_teks += 1                 # teks penutup setelah anak terakhir
            butir_id = ".".join(jalur + [f"t{nomor_teks}"])
            hasil.append(buat_passage(dok, bg, butir_id, x, konteks_induk, butir.hal, "butir"))
    return hasil


def passage_dari_bagian(dok: str, bg: Bagian) -> list[dict]:
    hasil, pengantar = [], []
    if bg.paragraf:
        kalimat = pecah_kalimat(gabung_baris([t for _, t in bg.paragraf]))
        if bg.butir and kalimat[-1].endswith(":"):
            pengantar = [kalimat.pop()]     # "... meliputi:" menjadi konteks butir di bawahnya
        for i, kelompok in enumerate(kelompokkan_kalimat(kalimat), start=1):
            hasil.append(buat_passage(dok, bg, f"p{i}", kelompok, [], bg.paragraf[0][0], "paragraf"))
    for butir in bg.butir:
        hasil += passage_dari_butir(dok, bg, butir, [], pengantar)
    return hasil


def passage_dari_tabel(dok: str, tabel: dict, bab: str = "", judul_bab: str = "",
                       subbab: str = "", judul_subbab: str = "") -> list[dict]:
    hasil = []
    for i, baris in enumerate(tabel["baris"], start=1):
        hasil.append({
            "id": f"{dok}-{tabel['id']}-{i}",
            "dok_id": dok,
            "halaman": tabel["halaman"],
            "bab": bab,
            "judul_bab": judul_bab,
            "subbab": subbab,
            "judul_subbab": judul_subbab,
            "butir": "",
            "konteks": tabel["judul"],
            "teks": tabel["templat"].format(**baris),
            "jenis": "tabel",
        })
    return hasil

Sekarang semua sumber digabung: passage dari parsing PDF, passage tabel Pedoman FILKOM (diberi BAB/subbab yang sama dengan lokasi tabelnya), serta isi dokumen `penilaian` yang diketik ulang.

In [15]:
def judul_dari_lokasi(dok: str, bab: str, kode: str) -> tuple[str, str]:
    bg = cari_bagian(dok, bab, kode)
    return bg.judul_bab, bg.judul_subbab


passages: list[dict] = []

# a) passage hasil parsing PDF
for dok in DOKUMEN_PDF:
    for bg in hasil_parse[dok]:
        passages += passage_dari_bagian(dok, bg)

# b) tabel yang dipotong dari PDF
for dok in DOKUMEN_PDF:
    for tabel in TRANSKRIPSI.get(dok, {}).get("tabel", []):
        judul_bab, judul_subbab = judul_dari_lokasi(dok, tabel["bab"], tabel["subbab"])
        passages += passage_dari_tabel(dok, tabel, tabel["bab"], judul_bab, tabel["subbab"], judul_subbab)

# c) dokumen yang diketik ulang manual
for dok, cfg in DOKUMEN.items():
    if cfg["metode"] != "manual":
        continue
    manual = TRANSKRIPSI[dok]
    for i, par in enumerate(manual.get("paragraf", []), start=1):
        passages.append({
            "id": f"{dok}-p{i}", "dok_id": dok, "halaman": par["halaman"],
            "bab": "", "judul_bab": "", "subbab": "", "judul_subbab": par["judul_subbab"],
            "butir": "", "konteks": "", "teks": par["teks"], "jenis": "paragraf",
        })
    for tabel in manual.get("tabel", []):
        passages += passage_dari_tabel(dok, tabel, judul_subbab=tabel["judul_subbab"])

print(f"Total passage: {len(passages)}")
print(Counter((p["dok_id"], p["jenis"]) for p in passages))

Total passage: 440
Counter({('pa2020', 'butir'): 354, ('pa2020', 'tabel'): 29, ('tatatertib', 'butir'): 26, ('penilaian', 'tabel'): 17, ('pa2020', 'paragraf'): 13, ('penilaian', 'paragraf'): 1})


Contoh passage: butir III.I.7 yang dibahas saat merancang format (syarat selesai kuliah 144 sks).

In [16]:
def tampilkan(p: dict) -> None:
    print(json.dumps(p, ensure_ascii=False, indent=2))


tampilkan(next(p for p in passages if "minimal 144 sks, dengan syarat" in p["teks"]))

{
  "id": "pa2020-II-I-7",
  "dok_id": "pa2020",
  "halaman": 21,
  "bab": "II",
  "judul_bab": "SISTEM PENDIDIKAN",
  "subbab": "I",
  "judul_subbab": "EVALUASI KEBERHASILAN STUDI",
  "butir": "7",
  "konteks": "",
  "teks": "7. Seorang mahasiswa dinyatakan telah selesai mengikuti kuliah pada suatu Program Studi bilamana telah mengumpulkan jumlah nilai kredit minimal 144 sks, dengan syarat-syarat sebagai berikut: a. IPK ≥ 2,00. b. Telah lulus seluruh mata kuliah wajib. c. Tidak ada nilai E. d. Total sks mata kuliah yang mempunyai nilai D dan D+, tidak melebihi 10% dari sks mata kuliah yang harus ditempuh. e. Telah menyelesaikan skripsi. f. Telah menyelesaikan tugas akademik lainnya. g. Telah lulus ujian akhir sarjana dan mengunggah skripsi ke repositori perguruan tinggi yang diintegrasikan dengan portal repositori Tugas Akhir Mahasiswa Kemenristekdikti (rama.ristekdikti.go.id), kecuali apabila dipublikasikan di jurnal. h. Telah memenuhi syarat-syarat administrasi. i. Memiliki nilai TO

In [17]:
print("Butir III.J.1 (> 150 kata) dipecah. Anak '1.' memakai konteks induk + kalimat pengantar 'Adapun ...':\n")
tampilkan(next(p for p in passages if p["id"] == "pa2020-III-J-1.1"))
print("\nKalimat '... meliputi:' menjadi konteks butir di bawahnya:\n")
tampilkan(next(p for p in passages if p["id"] == "pa2020-III-B-2"))
print("\nContoh passage tabel:\n")
tampilkan(next(p for p in passages if p["id"] == "pa2020-tabel1-2"))

Butir III.J.1 (> 150 kata) dipecah. Anak '1.' memakai konteks induk + kalimat pengantar 'Adapun ...':

{
  "id": "pa2020-III-J-1.1",
  "dok_id": "pa2020",
  "halaman": 29,
  "bab": "III",
  "judul_bab": "ADMINISTRASI PENDIDIKAN",
  "subbab": "J",
  "judul_subbab": "PERPINDAHAN MAHASISWA",
  "butir": "1.1",
  "konteks": "1. Perpindahan Mahasiswa di lingkungan Universitas Brawijaya Merupakan perpindahan mahasiswa antar program studi yang ada di lingkungan UB. Perpindahan antar program studi tersebut terdiri dari: › Adapun persyaratan dan prosedur yang harus dipenuhi dalam pengajuan perpindahan program studi di lingkungan UB antara lain:",
  "teks": "1. Mahasiswa yang dapat mengajukan pindah program studi adalah telah mengikuti pendidikan secara terus menerus sekurang-kurangnya 2 semester dan setinggi-tingginya 4 semester serta telah mengumpulkan: a. Untuk 2 semester, 24 sks dengan IPK sekurang-kurangnya 2,75. b. Untuk 4 semester, 48 sks dengan IPK sekurang-kurangnya 2,75.",
  "jenis": "b

## 8. Menyimpan hasil

- `data/passages.jsonl` — satu passage per baris (JSON Lines). Format ini dipilih karena teks penuh koma (`IPK 2,00`) yang mudah merusak CSV, dan cukup dibaca dengan `json.loads` per baris.
- `data/dokumen.json` — metadata tiap dokumen (judul, tahun, URL sumber). Metadata disimpan **sekali di sini**, tidak diulang di setiap passage; gabungkan lewat `dok_id`.

Cara membaca dari modul anggota lain:

```python
import json
passages = [json.loads(baris) for baris in open("data/passages.jsonl", encoding="utf-8")]
dokumen = json.load(open("data/dokumen.json", encoding="utf-8"))
```

In [18]:
path_passage = DIR_DATA / "passages.jsonl"
with path_passage.open("w", encoding="utf-8") as f:
    for p in passages:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

dokumen_json = {
    dok: {k: cfg[k] for k in ("judul", "tahun", "url", "file", "metode")}
    | ({"catatan": TRANSKRIPSI[dok]["catatan"]} if "catatan" in TRANSKRIPSI.get(dok, {}) else {})
    for dok, cfg in DOKUMEN.items()
}
path_dokumen = DIR_DATA / "dokumen.json"
path_dokumen.write_text(json.dumps(dokumen_json, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

for path in (path_passage, path_dokumen):
    print(f"{path.relative_to(DIR_KERJA)}  ({path.stat().st_size / 1024:.1f} KB)")

data\passages.jsonl  (214.4 KB)
data\dokumen.json  (1.3 KB)


## 9. Validasi

Validasi memastikan korpus siap dipakai anggota lain dan tidak ada informasi yang hilang di tengah jalan.

| Pemeriksaan | Tujuan |
|---|---|
| ID unik | `id` dipakai sebagai kunci ground truth oleh Anggota 5 |
| Distribusi panjang | passage terlalu pendek atau terlalu panjang mengganggu pencarian |
| Urutan penanda | sub-butir yang loncat (`a, b, d`) menandakan parsing salah |
| Angka tidak hilang | jumlah kemunculan `… sks` pada teks bersih = pada passage |
| Fakta kunci | fakta yang kemungkinan besar ditanyakan benar-benar ada dalam satu passage |
| Sampel acak | dibaca manual dan dibandingkan dengan PDF |

In [19]:
ids = [p["id"] for p in passages]
duplikat = [i for i, n in Counter(ids).items() if n > 1]
print("ID unik      :", "OK" if not duplikat else f"DUPLIKAT {duplikat}")

panjang = [jumlah_kata(p["teks"]) for p in passages]
print(f"Panjang kata : min {min(panjang)}, median {statistics.median(panjang)}, "
      f"rata-rata {statistics.mean(panjang):.1f}, maks {max(panjang)}")

batas = [(0, 10), (10, 25), (25, 50), (50, 100), (100, 150), (150, 10_000)]
for bawah, atas in batas:
    n = sum(bawah <= x < atas for x in panjang)
    label = f"{bawah}-{atas - 1}" if atas < 10_000 else f">={bawah}"
    print(f"  {label:>8} kata: {n:>3} {'█' * n}")

ID unik      : OK
Panjang kata : min 5, median 21.5, rata-rata 31.5, maks 145
       0-9 kata:  27 ███████████████████████████
     10-24 kata: 231 ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
     25-49 kata: 113 █████████████████████████████████████████████████████████████████████████████████████████████████████████████████
     50-99 kata:  51 ███████████████████████████████████████████████████
   100-149 kata:  18 ██████████████████
     >=150 kata:   0 


In [20]:
print(f"Passage > {BATAS_KATA} kata (butir tanpa anak, tidak bisa dipecah lagi):")
for p in passages:
    if jumlah_kata(p["teks"]) > BATAS_KATA:
        print(f"  {p['id']:<22} {jumlah_kata(p['teks'])} kata: {p['teks'][:80]}…")

print("\nPassage < 5 kata:")
for p in passages:
    if jumlah_kata(p["teks"]) < 5:
        print(f"  {p['id']:<22} {p['teks']!r}")

Passage > 150 kata (butir tanpa anak, tidak bisa dipecah lagi):



Passage < 5 kata:


In [21]:
ROMAWI = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x", "xi", "xii"]


HURUF = list("abcdefghijklmnopqrstuvwxyz")


def penanda_berikutnya(rangkaian: list[str]) -> set[str]:
    terakhir, n = rangkaian[-1], len(rangkaian)
    harapan = set()
    if terakhir.isdigit():
        harapan.add(str(int(terakhir) + 1))
    if rangkaian == ROMAWI[:n] and n < len(ROMAWI):
        harapan.add(ROMAWI[n])
    if rangkaian == HURUF[:n] and n < len(HURUF):
        harapan.add(HURUF[n])
    return harapan


def urutan_wajar(penanda: list[str]) -> bool:
    # Satu daftar boleh berisi beberapa rangkaian (mis. a, b lalu 1..11 pada III.J.1),
    # asalkan setiap rangkaian dimulai dari 1 / a / i dan berurutan.
    if penanda[0] not in ("1", "a", "i"):
        return False
    rangkaian = [penanda[0]]
    for p in penanda[1:]:
        if p in penanda_berikutnya(rangkaian):
            rangkaian.append(p)
        elif p in ("1", "a", "i"):
            rangkaian = [p]
        else:
            return False
    return True


def cek_urutan(daftar: list[Butir], lokasi: str, masalah: list[str]) -> None:
    if daftar and not urutan_wajar([b.penanda for b in daftar]):
        masalah.append(f"{lokasi}: {[b.penanda for b in daftar]}")
    for b in daftar:
        cek_urutan(b.anak, f"{lokasi}.{b.penanda}", masalah)


masalah_urutan = []
for dok in DOKUMEN_PDF:
    for bg in hasil_parse[dok]:
        cek_urutan(bg.butir, f"{dok} BAB {bg.bab or '-'} {bg.kode}", masalah_urutan)
print("Urutan penanda:", "OK, semua berurutan" if not masalah_urutan else "")
for m in masalah_urutan:
    print("  ", m)

Urutan penanda: OK, semua berurutan


In [22]:
POLA_SKS = re.compile(r"\b\d+\s*sks\b", re.IGNORECASE)

for dok in DOKUMEN_PDF:
    di_teks = sum(len(POLA_SKS.findall(b.teks)) for b in baris_final[dok])
    milik_dok = [p for p in passages if p["dok_id"] == dok and p["jenis"] != "tabel"]
    segmen = {p["teks"] for p in milik_dok} | {k for p in milik_dok for k in p["konteks"].split(" › ") if k}
    di_passage = sum(len(POLA_SKS.findall(s)) for s in segmen)
    status = "OK" if di_teks == di_passage else "BERBEDA"
    print(f"{dok:<11} kemunculan '… sks': teks bersih {di_teks}, passage {di_passage} -> {status}")

pa2020      kemunculan '… sks': teks bersih 40, passage 40 -> OK
tatatertib  kemunculan '… sks': teks bersih 0, passage 0 -> OK


In [23]:
FAKTA_KUNCI = {
    "syarat selesai kuliah 144 sks": r"minimal 144 sks",
    "batas masa studi 14 semester": r"paling lama empat belas semester",
    "IPK minimal selesai kuliah": r"a\. IPK ≥ 2,00",
    "syarat IPK alih program": r"Mempunyai IPK ≥ 3,0",
    "IPK tugas belajar alih program": r"IPK ≥ 2,50",
    "skor TOEIC kelulusan": r"TOEIC dengan skor 500",
    "durasi ujian skripsi": r"paling lama 75 menit",
    "kehadiran minimal UAS": r"minimal 80%",
    "batas terlambat ujian": r"maksimal 15 \(lima belas\) menit",
    "beban semester antara": r"paling banyak 9 sks",
    "FILKOM resmi berdiri": r"PTIIK secara resmi menjadi Fakultas Ilmu Komputer",
    "predikat cum laude": r"Dengan Pujian \(Cum Laude\)",
    "konversi B+": r"75 < N ≤ 80 dikonversi menjadi nilai huruf B\+",
    "bobot nilai K (PTIIK)": r"nilai huruf K memiliki bobot 0",
    "rumus Indeks Prestasi": r"IP = Σ \(K_i × Na_i\)",
    "rumus Nilai Akhir": r"NA = \(Σ Bk_i × Nk_i",
}

for nama, pola in FAKTA_KUNCI.items():
    cocok = [p["id"] for p in passages if re.search(pola, p["teks"])]
    print(f"{'✓' if cocok else '✗'} {nama:<32} {cocok[:3]}")

✓ syarat selesai kuliah 144 sks    ['pa2020-II-I-7']
✓ batas masa studi 14 semester     ['pa2020-II-I-8']
✓ IPK minimal selesai kuliah       ['pa2020-II-I-7']
✓ syarat IPK alih program          ['pa2020-III-H-3']
✓ IPK tugas belajar alih program   ['pa2020-III-H-3']
✓ skor TOEIC kelulusan             ['pa2020-II-I-7']
✓ durasi ujian skripsi             ['pa2020-VI-D-10']
✓ kehadiran minimal UAS            ['pa2020-II-F-9']
✓ batas terlambat ujian            ['tatatertib-X-S1-3']
✓ beban semester antara            ['pa2020-II-J-11']
✓ FILKOM resmi berdiri             ['pa2020-X-S5-p3']
✓ predikat cum laude               ['pa2020-VI-E-3']
✓ konversi B+                      ['pa2020-tabel3-2', 'penilaian-konversi-2']
✓ bobot nilai K (PTIIK)            ['penilaian-bobot-9']
✓ rumus Indeks Prestasi            ['pa2020-II-H-3']
✓ rumus Nilai Akhir                ['pa2020-II-G-4']


In [24]:
random.seed(42)
print("Sampel acak untuk dicek manual terhadap PDF:\n")
for p in random.sample(passages, 5):
    print(f"[{p['id']}] hlm. {p['halaman']} | {p['judul_subbab'][:50]}")
    if p["konteks"]:
        print(f"  konteks: {p['konteks'][:120]}")
    print(f"  teks   : {p['teks'][:300]}\n")

Sampel acak untuk dicek manual terhadap PDF:

[pa2020-VI-C-1] hlm. 42 | SEMINAR HASIL
  teks   : 1. Seminar Hasil adalah kegiatan yang harus dilakukan oleh seorang mahasiswa yang telah menuntaskan skripsinya dalam bentuk presentasi hasil-hasil penelitian skripsi yang telah dilakukan di depan Dosen Fasilitator dan sejumlah mahasiswa yang lain.

[pa2020-II-D-4] hlm. 15 | PELAKSANAAN PEMBELAJARAN
  teks   : 4. Untuk meningkatkan mutu pelaksanaan pengajaran, masing-masing Program Studi dibantu oleh kelompok dosen (Kelompok Jabatan Fungsional Dosen) dalam hal pengembangan dan pengelolaan keilmuan, serta Unit Jaminan Mutu dalam hal manajemen penyelenggaraannya.

[pa2020-I-0-p1] hlm. 11 | 
  teks   : Untuk memperjelas konteks dan ruang lingkup pembahasan pedoman akademik di dalam buku ini, maka terdapat beberapa istilah dan ketentuan yang perlu diperhatikan. Istilah dan ketentuan yang termuat dalam Bab ini akan menjadi acuan konteks dalam penggunaan istilah serta ketentuan di bab-bab selanjut

## Ringkasan dan catatan untuk anggota lain

**Hasil:** `data/passages.jsonl` dan `data/dokumen.json` siap dipakai. Jalankan sel di bawah untuk ringkasan angka terbaru.

**Catatan penting:**

1. **Anggota 2 & 3** — `teks` disimpan apa adanya (huruf besar/kecil, angka, tanda baca). Lakukan *preprocessing* (case folding, tokenisasi, stemming, stopword) dengan **satu fungsi yang sama** untuk pertanyaan dan passage. Field `konteks` dan `judul_subbab` sebaiknya ikut di-index, karena passage hasil pemecahan bergantung pada konteksnya.
2. **Anggota 4** — angka muncul dalam dua bentuk: digit (`144 sks`, `IPK ≥ 2,00`) dan kata (`empat belas semester`, `15 (lima belas) menit`). Desimal memakai **koma** (`2,00`, `3,5`). Simbol pembanding yang dipakai: `≥ ≤ < >`.
3. **Anggota 5** — setiap pertanyaan ground truth sebaiknya mencatat `id` passage yang memuat jawabannya, supaya evaluasi *retrieval* (Precision@k / MRR) bisa dihitung.
4. **Batasan korpus** — aturan mengacu ke **Pedoman FILKOM 2020**, Tata Tertib ujian, dan Pedoman Penilaian PTIIK 2015. Informasi yang sama dari dua dokumen (konversi nilai dan bobot) sengaja tetap disimpan dengan sumbernya masing-masing.
5. **Keputusan manual** yang perlu disebut di laporan: transkripsi 4 tabel dan 2 rumus (Nilai Akhir, Indeks Prestasi) Pedoman FILKOM, serta seluruh dokumen Pedoman Penilaian (hasil scan), termasuk bobot nilai D yang tertulis `14` di dokumen scan dan dicatat sebagai `1`.

In [25]:
ringkasan = Counter(p["dok_id"] for p in passages)
print(f"Total passage : {len(passages)}")
for dok, n in ringkasan.items():
    print(f"  {dok:<11} {n:>4} passage  ({DOKUMEN[dok]['judul']}, {DOKUMEN[dok]['tahun']})")
print(f"Jenis         : {dict(Counter(p['jenis'] for p in passages))}")
print(f"Total kata    : {sum(jumlah_kata(p['teks']) for p in passages):,}")

Total passage : 440
  pa2020       396 passage  (Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya, 2020)
  tatatertib    26 passage  (Tata Tertib Pelaksanaan Ujian (UTS & UAS) FILKOM UB, 2016)
  penilaian     18 passage  (Pedoman Penilaian dan Perhitungan Indeks Prestasi PTIIK, 2015)
Jenis         : {'paragraf': 14, 'butir': 380, 'tabel': 46}
Total kata    : 13,856
